In [1]:
import model.create_search_index
import pandas as pd
import numpy as np
from main import SearchSystem

In [ ]:
embed_model_small = "intfloat/multilingual-e5-small"
embed_model_base = "intfloat/multilingual-e5-base"
embed_model_large = "Alibaba-NLP/gte-multilingual-base"




#Проекция данных на 2d

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap
import hdbscan
import warnings
warnings.filterwarnings('ignore')

# 1. Загрузка эмбеддингов
# Предположим, файл имеет формат .npy (массив numpy)
# Если у вас другой формат (csv, txt), раскомментируйте соответствующий блок
def load_embeddings(file_path):
    # Пример для .npy
    embeddings = np.load(file_path)
    print(f"Загружено {embeddings.shape[0]} эмбеддингов размерностью {embeddings.shape[1]}")
    return embeddings

# Альтернативно для CSV (без заголовка, каждая строка - вектор)
# import pandas as pd
# df = pd.read_csv(file_path, header=None)
# embeddings = df.values

file_path = "../models/embeddings/intfloat_multilingual-e5-small.npy"  # укажите ваш путь
embeddings = load_embeddings(file_path)

# 2. Применение UMAP для снижения размерности до 2D (или 3D)
# Параметры можно подбирать: n_neighbors, min_dist
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding_2d = reducer.fit_transform(embeddings)
print("Размерность после UMAP:", embedding_2d.shape)

# 3. Визуализация проекции UMAP (базовая)
plt.figure(figsize=(10, 8))
plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], s=5, alpha=0.6)
plt.title("UMAP проекция эмбеддингов")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.show()

# 4. Оценка количества кластеров
# Можно использовать несколько методов:

# 4.1. HDBSCAN на UMAP-проекции (автоматическое определение кластеров)
clusterer = hdbscan.HDBSCAN(min_cluster_size=10, gen_min_span_tree=True)
cluster_labels = clusterer.fit_predict(embedding_2d)
n_clusters_hdbscan = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
print(f"Число кластеров по HDBSCAN: {n_clusters_hdbscan} (шумовые точки помечены -1)")

# Визуализация с раскраской по кластерам HDBSCAN
plt.figure(figsize=(10, 8))
palette = sns.color_palette('deep', n_clusters_hdbscan)
colors = [palette[x] if x != -1 else (0.5,0.5,0.5) for x in cluster_labels]
plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=colors, s=5, alpha=0.7)
plt.title(f"Кластеризация HDBSCAN (кластеров: {n_clusters_hdbscan})")
plt.show()

# 4.2. Метод локтя (Elbow) для KMeans на исходных эмбеддингах
# (или на UMAP, но лучше на исходных для оценки структуры)
inertia = []
silhouette_scores = []
K_range = range(2, 30)  # примерный диапазон, можно расширить

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)
    inertia.append(kmeans.inertia_)
    # силуэт считаем на подвыборке, если данных много (>10000)
    if len(embeddings) > 10000:
        sample_idx = np.random.choice(len(embeddings), size=10000, replace=False)
        sil = silhouette_score(embeddings[sample_idx], labels[sample_idx])
    else:
        sil = silhouette_score(embeddings, labels)
    silhouette_scores.append(sil)

# Построение графиков для выбора k
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(K_range, inertia, 'bo-')
ax1.set_xlabel('Количество кластеров k')
ax1.set_ylabel('Интерция (сумма квадратов расстояний)')
ax1.set_title('Метод локтя')

ax2.plot(K_range, silhouette_scores, 'ro-')
ax2.set_xlabel('Количество кластеров k')
ax2.set_ylabel('Силуэт')
ax2.set_title('Оценка силуэта')
plt.tight_layout()
plt.show()

# 4.3. Анализ результатов и рекомендация
print("\nРекомендации по выбору количества кластеров для FAISS IVF:")
print("- HDBSCAN дал {} кластеров (не учитывая шум)".format(n_clusters_hdbscan))
print("- По методу локтя и силуэту оптимальное k может находиться в районе излома графика.")
print("\nДля FAISS IVF число кластеров обычно выбирают как компромисс между точностью и скоростью.")
print("Типичные значения: nlist = 4*sqrt(N) до 16*sqrt(N), где N - число векторов.")
print(f"Для вашего N={len(embeddings)} это примерно от {int(4*np.sqrt(len(embeddings)))} до {int(16*np.sqrt(len(embeddings)))}.")
print("Рекомендуется взять число, близкое к полученному из анализа (например, {}), но в пределах разумного для FAISS (обычно 100-1000).".format(n_clusters_hdbscan))

Загружено 244484 эмбеддингов размерностью 384


In [11]:
from __future__ import annotations

import argparse
import sys
from pathlib import Path
from typing import Any, List, Tuple

import faiss
import numpy as np
import torch
import yaml
from sentence_transformers import SentenceTransformer

CONFIG_PATH = "../config/ml_config.yaml"

with open(CONFIG_PATH, encoding="utf-8") as f:
    config = yaml.safe_load(f)

available_models = config["available_models"]


small = SearchSystem("intfloat/multilingual-e5-small", 
                     Path("../models/indexes/faiss_flat_(metric=ip)__intfloat_multilingual-e5-small.index"),
                     Path("../models/embeddings/cte_ids.npy"),
                     available_models=available_models,
                    )

base = SearchSystem("intfloat/multilingual-e5-base", 
                     Path("../models/indexes/faiss_flat_(metric=ip)__intfloat_multilingual-e5-base.index"),
                     Path("../models/embeddings/cte_ids.npy"),
                     available_models=available_models,
                    )

Initialized model: intfloat/multilingual-e5-small on cpu
Loaded FAISS index from: ../models/indexes/faiss_flat_(metric=ip)__intfloat_multilingual-e5-small.index
Loaded CTE ids from: ../models/embeddings/cte_ids.npy (shape=(244484,))
Initialized model: intfloat/multilingual-e5-base on cpu
Loaded FAISS index from: ../models/indexes/faiss_flat_(metric=ip)__intfloat_multilingual-e5-base.index
Loaded CTE ids from: ../models/embeddings/cte_ids.npy (shape=(244484,))


In [12]:
import pandas as pd
import time 

cte_df = pd.read_csv("../data/prep/CTE.csv")
top_k = config["search_top_k"]

cte_df

,CTE_id,CTE_name,category,manufacturer,characteristics
0,34863000,Мусорные пакеты 35л,Пакеты полимерные,ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ СПРИН...,"Количество в упаковке:30.00000;Толщина, мкм:50..."
1,20647780,Пакеты для мусора,Пакеты полимерные,ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ СПРИН...,Цвет:черный;Ширина:470.00000;Количество рулоно...
2,34865337,Мусорные пакеты 240л,Пакеты полимерные,ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ СПРИН...,Количество в упаковке:10.00000;Цвет:черный;Тол...
3,38669978,"Радиостанция носимая аналоговая Шеврон, ""АЙТИ-...",Радиостанции цифровые,"ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ ""АЙТИ...",Максимальная рабочая температура:50.00000;Коли...
4,38515747,Набор реагентов для иммуноферментного определе...,Наборы реактивов/реагентов,"ООО ""ХЕМА""",Концентрация аналита в наибольшей калибровочно...
...,...,...,...,...,...
244479,39163435,Гофра для раковины EWRIKA 85001 85036-750 40 м...,Удлинители гофрированные для оборудования сант...,EWRIKA,Ширина:65.00000;Комплектация:Комплект прокладо...
244480,39163497,"Творог Б.Ю. Александров 9%, 180г",Творог,Б.Ю. АЛЕКСАНДРОВ,Наличие вкусовых компонентов:Нет;Вид молочного...
244481,39165297,"Труба VALFEX OPTIMA ф110, с раструбом, L=0.25 ...",Детали и фитинги трубопроводов из полимеров,VALFEX OPTIMA,Материал трубы:полипропилен;Вид товара:Труба;Д...
244482,1374499,Мука пшеничная Макфа весовая,Мука пшеничная хлебопекарная,"АО ""Макфа""",Назначение:пшеничная хлебопекарная;Тарная упак...


In [29]:
def search(query:str):
    print(f"Query: {query}\n{"_"*50}")
    print(f"small model:")
    small_start_time = time.perf_counter()
    small_ids, small_sims = small.search(query, top_k=top_k)
    small_end_time = time.perf_counter()
    for result, sim in zip(small_ids, small_sims):
        row = cte_df[cte_df["CTE_id"] == result]
        print(f"\t({sim:.4f}) {row["CTE_name"].iloc[0]} {row["category"].iloc[0]} {row["manufacturer"].iloc[0]} {row["characteristics"].iloc[0]}\n")
    print(f"Runtime result: {small_end_time - small_start_time}")
    
    print(f"\nbase model:")
    base_start_time = time.perf_counter()
    base_ids, base_sims = base.search(query, top_k=top_k)
    base_end_time = time.perf_counter()
    for result, sim in zip(base_ids, base_sims):
        row = cte_df[cte_df["CTE_id"] == result]
        print(f"\t({sim:.4f}) {row["CTE_name"].iloc[0]} {row["category"].iloc[0]} {row["manufacturer"].iloc[0]} {row["characteristics"].iloc[0]}\n")        
    print(f"Runtime result: {base_end_time - base_start_time}")
    print("="*50)

In [33]:
query = "шприц 10мл"

search(query)

Query: шприц 10мл
__________________________________________________
small model:
	(0.9029) Шприц 10мл 3-х компонентный с иглой Шприц общего назначения Shandong Zibo Shanchuan Medical Intrument Co., Ltd. Упаковка:Индивидуальная, стерильная;Количество в упаковке:100;Калибр иглы:21;Тип соединения с иглой:Луер-слип

	(0.8934) Шприц инъекционный 10 мл (21G x 1/2 0.8x40) Шприц общего назначения Avanti Medical Вид расходные материалы для парентерального питания и инфузионной терапии:Шприц общего назначения;Вид товаров:Медицинские товары;Градуированный объем шприца:10.00000;Игла в комплекте:1;Минимальный остаточный срок годности на дату поставки:12.00000;Требования к упаковке:Стерильная;Длина иглы:40.00000;Диаметр иглы:0.80000;Вид расходные материалы для парентерального и энтерального введения, инфузионной терапии и инъекций:Расходные материалы для парентерального питания и инфузионной терапии;Вид товары медицинские:Медицинские расходные материалы;Вид продукции:Товары;Тип шприца:Трехкомпонент